# PySpark DataFrame Data Engineering Project — Healthcare Analytics

Business-oriented PySpark DataFrame project using two related input files.

**Input files:** `patients.csv`, `visits.csv`

**Questions:** 30

Topics include DataFrame creation, schema, select, filter, withColumn, when, cast, groupBy, agg, joins, orderBy, window functions, null handling, duplicates, date functions, pivot, union-style reporting, and business KPIs.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("PySparkDataFrameProject").getOrCreate()

df1 = spark.read.option("header", True).option("inferSchema", True).csv("patients.csv")
df2 = spark.read.option("header", True).option("inferSchema", True).csv("visits.csv")

df1.show(5, truncate=False)
df2.show(5, truncate=False)


## Q1. Inspect Schemas

Display both DataFrame schemas and row counts.

In [ ]:
df1.printSchema()
df2.printSchema()
print(df1.count(), df2.count())

## Q2. Select Business Columns

Select patient_id and city from the first DataFrame.

In [ ]:
df1.select('patient_id','city').show(20, truncate=False)

## Q3. Filter Records

Filter the second DataFrame where treatment_cost is greater than its average.

In [ ]:
avg_value = df2.select(avg('treatment_cost').alias('avg')).first()['avg']
df2.filter(col('treatment_cost') > avg_value).show(20, truncate=False)

## Q4. Create Derived Column

Create a new column based on treatment_cost using a High/Normal classification.

In [ ]:
df2.withColumn('value_category', when(col('treatment_cost') >= avg('treatment_cost').over(Window.partitionBy()), 'High').otherwise('Normal')).show(20, truncate=False)

## Q5. Null Analysis

Show null counts for every column in both DataFrames.

In [ ]:
df1.select([sum(col(c).isNull().cast('int')).alias(c) for c in df1.columns]).show()
df2.select([sum(col(c).isNull().cast('int')).alias(c) for c in df2.columns]).show()

## Q6. Handle Null Values

Fill null values in city and department with 'Unknown'.

In [ ]:
df1.fillna({'city': 'Unknown'}).show(5)
df2.fillna({'department': 'Unknown'}).show(5)

## Q7. Remove Duplicates

Remove duplicate records and compare row counts.

In [ ]:
print('Before:', df1.count(), df2.count())
print('After:', df1.dropDuplicates().count(), df2.dropDuplicates().count())

## Q8. Inner Join

Join the two DataFrames using patient_id and patient_id.

In [ ]:
joined = df1.join(df2, df1['patient_id'] == df2['patient_id'], 'inner')
joined.show(20, truncate=False)

## Q9. Left Join

Perform a left join and identify unmatched records.

In [ ]:
left = df1.join(df2, df1['patient_id'] == df2['patient_id'], 'left')
left.show(20, truncate=False)

## Q10. Group By Category

Calculate count and average treatment_cost by department.

In [ ]:
df2.groupBy('department').agg(count('*').alias('records'), avg('treatment_cost').alias('avg_value')).orderBy(desc('avg_value')).show()

## Q11. Total by Category

Calculate total treatment_cost by department.

In [ ]:
df2.groupBy('department').agg(sum('treatment_cost').alias('total_value')).orderBy(desc('total_value')).show()

## Q12. Maximum by Category

Find maximum treatment_cost in each department.

In [ ]:
df2.groupBy('department').agg(max('treatment_cost').alias('max_value')).show()

## Q13. Minimum by Category

Find minimum treatment_cost in each department.

In [ ]:
df2.groupBy('department').agg(min('treatment_cost').alias('min_value')).show()

## Q14. Having Condition

Show department groups whose total treatment_cost is greater than the overall average treatment_cost.

In [ ]:
overall = df2.select(avg('treatment_cost').alias('v')).first()['v']
df2.groupBy('department').agg(sum('treatment_cost').alias('total')).filter(col('total') > overall).show()

## Q15. Top 10 Records

Find the top 10 records by treatment_cost.

In [ ]:
df2.orderBy(desc('treatment_cost')).limit(10).show(truncate=False)

## Q16. Bottom 10 Records

Find the bottom 10 records by treatment_cost.

In [ ]:
df2.orderBy(asc('treatment_cost')).limit(10).show(truncate=False)

## Q17. Window Rank

Rank records by treatment_cost within each department.

In [ ]:
w = Window.partitionBy('department').orderBy(desc('treatment_cost'))
df2.withColumn('rank', dense_rank().over(w)).show(30, truncate=False)

## Q18. Running Total

Calculate a running total of treatment_cost within each department.

In [ ]:
w = Window.partitionBy('department').orderBy('visit_date').rowsBetween(Window.unboundedPreceding, Window.currentRow)
df2.withColumn('running_total', sum('treatment_cost').over(w)).show(30, truncate=False)

## Q19. Previous Value

Use lag to compare each record's treatment_cost with the previous record in its department.

In [ ]:
w = Window.partitionBy('department').orderBy('visit_date')
df2.withColumn('previous_value', lag('treatment_cost').over(w)).show(30, truncate=False)

## Q20. Percentage Contribution

Calculate each department's contribution to total treatment_cost.

In [ ]:
tot = df2.select(sum('treatment_cost').alias('t')).first()['t']
df2.groupBy('department').agg(sum('treatment_cost').alias('total')).withColumn('pct', col('total')/lit(tot)*100).orderBy(desc('pct')).show()

## Q21. Date Extraction

Extract year and month from visit_date.

In [ ]:
df2.withColumn('year', year('visit_date')).withColumn('month', month('visit_date')).show(20, truncate=False)

## Q22. Monthly KPI

Calculate total and average treatment_cost by month.

In [ ]:
df2.withColumn('month', date_format('visit_date','yyyy-MM')).groupBy('month').agg(sum('treatment_cost').alias('total'), avg('treatment_cost').alias('average')).orderBy('month').show()

## Q23. Pivot Report

Create a pivot report with department as rows and year of visit_date as columns.

In [ ]:
df2.withColumn('year', year('visit_date')).groupBy('department').pivot('year').agg(sum('treatment_cost')).show()

## Q24. Conditional KPI

Count records where treatment_cost is above the category average.

In [ ]:
w=Window.partitionBy('department')
df2.withColumn('cat_avg',avg('treatment_cost').over(w)).filter(col('treatment_cost')>col('cat_avg')).groupBy('department').count().show()

## Q25. Join Aggregation

Join the DataFrames and calculate total treatment_cost by city.

In [ ]:
df1.join(df2, df1['patient_id']==df2['patient_id']).groupBy(df1['city']).agg(sum(df2['treatment_cost']).alias('total')).orderBy(desc('total')).show()

## Q26. Join Ranking

After joining, rank city by total treatment_cost.

In [ ]:
r=df1.join(df2,df1['patient_id']==df2['patient_id']).groupBy(df1['city']).agg(sum(df2['treatment_cost']).alias('total'))
r.withColumn('rank',dense_rank().over(Window.orderBy(desc('total')))).show()

## Q27. Business Segmentation

Create Low, Medium and High segments for treatment_cost.

In [ ]:
df2.withColumn('segment',when(col('treatment_cost')<1000,'Low').when(col('treatment_cost')<5000,'Medium').otherwise('High')).groupBy('segment').count().show()

## Q28. Data Quality Check

Count invalid negative treatment_cost values.

In [ ]:
df2.filter(col('treatment_cost')<0).count()

## Q29. Final Business Report

Create a final report by department with count, total, average, minimum, maximum and rank.

In [ ]:
r=df2.groupBy('department').agg(count('*').alias('records'),sum('treatment_cost').alias('total'),avg('treatment_cost').alias('average'),min('treatment_cost').alias('minimum'),max('treatment_cost').alias('maximum'))
r.withColumn('rank',dense_rank().over(Window.orderBy(desc('total')))).orderBy('rank').show(truncate=False)

## End of Project